Imports

In [9]:
from pathlib import Path
import pandas as pd

Rutas

In [10]:
BASE           = Path("../../")
IMP_DIR        = BASE / "models" / "importances"
DASHBOARD_DATA = BASE / "dashboard" / "data"
DASHBOARD_DATA.mkdir(parents=True, exist_ok=True)
print("Importances dir:",   IMP_DIR)
print("Dashboard data dir:", DASHBOARD_DATA)

Importances dir: ..\..\models\importances
Dashboard data dir: ..\..\dashboard\data


Cargar importancias de todos los modelos

In [11]:
files = {
    "RandomForest": IMP_DIR / "random_forest_importance_features.csv",
    "SARIMAX":      IMP_DIR / "sarimax_importance_features.csv",
    "XGBoost":      IMP_DIR / "xgb_importance_features.csv",
}
dfs = []
for model_name, path in files.items():
    if path.exists():
        df = pd.read_csv(path)
        df["modelo"] = model_name
        dfs.append(df)
    else:
        print(f"⚠️ No existe archivo: {path}")

df_imp = pd.concat(dfs, ignore_index=True)
df_imp.head()

,feature,weight,sign_value,sign,hotel,modelo,coef,ci_low,ci_high,p_value,signo,peso_abs,method
0,rn_B,0.063802,0.038914,positivo,HOTEL_1,RandomForest,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rn_WEL,0.059517,0.025016,positivo,HOTEL_1,RandomForest,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,rn_H,0.040259,-0.031393,negativo,HOTEL_1,RandomForest,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,rn_K,0.027779,0.006873,positivo,HOTEL_1,RandomForest,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,rn_T,0.008354,-0.006400,negativo,HOTEL_1,RandomForest,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Normalizar columnas y asegurar consistencia

In [12]:
df_imp.columns = df_imp.columns.str.lower()

df_imp["weight"]     = df_imp.get("weight",     df_imp.get("peso_abs"))
df_imp["sign_value"] = df_imp.get("sign_value", df_imp.get("coef_1sigma"))
df_imp["sign"]       = df_imp["sign"].astype(str)

df_imp_standard = df_imp[
    ["hotel", "modelo", "feature", "weight", "sign", "sign_value"]
].copy()
df_imp_standard.head()

,hotel,modelo,feature,weight,sign,sign_value
0,HOTEL_1,RandomForest,rn_B,0.063802,positivo,0.038914
1,HOTEL_1,RandomForest,rn_WEL,0.059517,positivo,0.025016
2,HOTEL_1,RandomForest,rn_H,0.040259,negativo,-0.031393
3,HOTEL_1,RandomForest,rn_K,0.027779,positivo,0.006873
4,HOTEL_1,RandomForest,rn_T,0.008354,negativo,-0.006400


Crear ranking de importancia (por hotel-modelo)

In [13]:
df_imp_standard["rank"] = (
    df_imp_standard.groupby(["hotel", "modelo"])["weight"]
                   .rank(ascending=False, method="first")
)
df_imp_standard.sort_values(["hotel", "modelo", "rank"]).head()

,hotel,modelo,feature,weight,sign,sign_value,rank
0,HOTEL_1,RandomForest,rn_B,0.063802,positivo,0.038914,1.0
1,HOTEL_1,RandomForest,rn_WEL,0.059517,positivo,0.025016,2.0
2,HOTEL_1,RandomForest,rn_H,0.040259,negativo,-0.031393,3.0
3,HOTEL_1,RandomForest,rn_K,0.027779,positivo,0.006873,4.0
4,HOTEL_1,RandomForest,rn_T,0.008354,negativo,-0.006400,5.0


(Opcional pero recomendado) Clasificar features por “familia”

In [14]:

def feature_family(name: str) -> str:
    if name.startswith("ocup_"):          return "ocupación_histórica"
    if name.startswith("rn_total_ttoo"):  return "total_ttoo"
    if name.startswith("rn_J"):           return "J"
    if name.startswith("rn_T"):           return "T"
    if name.startswith("rn_WEL"):         return "WEL"
    if name.startswith("rn_B"):           return "B"
    if name.startswith("rn_"):            return "otros_ttoo"
    if name.startswith("share_"):         return "shares"
    if name in ["dow", "is_weekend", "month", "weekofyear", "dayofyear"]:
        return "calendario"
    if name in ["stock", "adr"]:          return "hotel_info"
    return "otros"

df_imp_standard["family"] = df_imp_standard["feature"].apply(feature_family)

Familia_Display dinámica por hotel

In [15]:
def asignar_familia_display(df, modelo="XGBoost", top_n=4):
    ttoo_families = ["T", "J", "B", "WEL", "otros_ttoo"]
    result_rows   = []

    for hotel in df["hotel"].unique():
        df_ttoo_hotel = df[
            (df["hotel"]  == hotel)  &
            (df["modelo"] == modelo) &
            (df["family"].isin(ttoo_families))
        ].copy()

        top_features = (
            df_ttoo_hotel
            .sort_values("weight", ascending=False)
            .head(top_n)["feature"]
            .tolist()
        )
        print(f"\n✅ Top {top_n} touroperadores para {hotel}:")
        print(
            df_ttoo_hotel
            .sort_values("weight", ascending=False)
            .head(top_n)[["feature", "weight"]]
            .to_string(index=False)
        )

        for _, row in df[df["hotel"] == hotel].iterrows():
            if row["family"] in ["season", "otros"]:
                familia = "Estación"
            elif row["family"] == "calendario":
                familia = "Calendario"
            elif row["feature"] in top_features:
                familia = row["feature"].replace("rn_", "")
            else:
                familia = "Otros touroperadores"
            result_rows.append({**row.to_dict(), "Familia_Display": familia})

    return pd.DataFrame(result_rows)

df_imp_standard = asignar_familia_display(df_imp_standard)

print("\nFamilias por hotel:")
display(
    df_imp_standard[df_imp_standard["modelo"] == "XGBoost"]
    .groupby(["hotel", "Familia_Display"])["feature"]
    .count()
    .reset_index()
    .rename(columns={"feature": "n_features"})
)


✅ Top 4 touroperadores para HOTEL_1:
feature   weight
   rn_B 0.064482
 rn_WEL 0.058270
   rn_H 0.045132
   rn_K 0.034282

✅ Top 4 touroperadores para HOTEL_2:
feature   weight
   rn_T 0.092299
 rn_WEL 0.026346
   rn_J 0.022745
   rn_B 0.017714

✅ Top 4 touroperadores para HOTEL_3:
feature   weight
   rn_J 0.051967
 rn_WEL 0.018073
  rn_AV 0.016656
   rn_T 0.015432

Familias por hotel:


,hotel,Familia_Display,n_features
0,HOTEL_1,B,1
1,HOTEL_1,Calendario,1
2,HOTEL_1,Estación,3
3,HOTEL_1,H,1
4,HOTEL_1,K,1
5,HOTEL_1,Otros touroperadores,12
6,HOTEL_1,WEL,1
7,HOTEL_2,B,1
8,HOTEL_2,Calendario,1
9,HOTEL_2,Estación,3


Guardar dataset final

In [16]:
output_file = DASHBOARD_DATA / "feature_importance.parquet"
df_imp_standard.to_parquet(output_file, index=False)
print("✅ feature_importance.parquet guardado en:", output_file)
df_imp_standard.head()

✅ feature_importance.parquet guardado en: ..\..\dashboard\data\feature_importance.parquet


,hotel,modelo,feature,weight,sign,sign_value,rank,family,Familia_Display
0,HOTEL_1,RandomForest,rn_B,0.063802,positivo,0.038914,1.0,B,B
1,HOTEL_1,RandomForest,rn_WEL,0.059517,positivo,0.025016,2.0,WEL,WEL
2,HOTEL_1,RandomForest,rn_H,0.040259,negativo,-0.031393,3.0,otros_ttoo,H
3,HOTEL_1,RandomForest,rn_K,0.027779,positivo,0.006873,4.0,otros_ttoo,K
4,HOTEL_1,RandomForest,rn_T,0.008354,negativo,-0.006400,5.0,T,Otros touroperadores
